# Bank Marketing 补充实验：公式、参数与可执行代码

本文件配合 `Bank_Marketing_Assignment.ipynb` 阅读，集中保留主 Notebook 第7–10节及扩展算法附录的实验实现。
**只需要本文件、原始 `bank-additional-full.csv` 和声明的 Python 包；不需要历史模型、预测文件、备份 ZIP 或项目脚本。**
这里的表格由下面的代码计算；没有把历史成绩写成重新运行的输出。

覆盖17类实际执行过的实验。主 Notebook 中“关闭阶段汇总”“已有宏观/个人宏观补测”和“联系LR”重复引用已有模型，
不是另外的算法；相应实现分别位于本文件的诊断、宏观、个人/宏观、信息范围和Precision约束章节。
原主 Notebook 保留实验结果，本文件承担方法说明和重新运行。

## 0. 怎么运行

1. 将两个 Notebook 与原始 CSV 放在同一目录，选择声明依赖所在的 Python 内核。
2. 默认 `EXPERIMENTS=set()`、`RUN_TEST=False`。Run All只加载开发数据、定义函数并展示参数，不训练模型，不读test。
3. 在第1节把 `EXPERIMENTS` 改成要运行的支线，例如 `{"refinement", "window", "kalman"}`；
   `FOLD_NAMES=["val"]` 可以只重现最终训练窗。默认五折才能得到完整的前窗阈值迁移对照。
4. `MODEL_FILTER` 可限定同一支线的已有配置；设为空字典表示全部。要运行全部开发实验，
   使用 `EXPERIMENTS=set(FAMILY_NAMES)`。这会触发大量串行拟合，不是快速示例。
5. 最后一节的 `RUN_TEST=True` 只用于固定开发方案的补充测试。它不会重新拟合基模型、校准器或阈值，
   不进行test标签反馈更新。不要依据结果新增候选后再声称独立确认。
6. 运行产生的模型与CSV自动保存在 `OUTPUT_DIR` 指定目录；它们是本次运行的输出，不是运行前要准备的输入。
   精简仓库可以不提交这些生成文件。每次独立重跑可修改 `OUTPUT_DIR`，避免覆盖自己的旧结果。
   长参数表先显示10行，其余行可展开；所有列和配置都保留在输出中。

**可复现范围与历史证据：** 默认重跑产生的是当前内核下的新结果。历史test已被查看，
所以test评价仍属于事后补充。依赖版本、浮点实现和求解收敛可能造成数值差异；
不能把一张看起来相近的表当作逐样本一致的证明。
本文件未加入最初建议但没有实际执行的 isotonic 校准，也没有为“改进结果”增加新参数网格。


### 0.1 依赖

整理与开发验证使用Python 3.11.9；历史主要版本如下。
安装命令需要在所选Notebook内核的环境中执行，不会由本文件自动安装。

| 包 | 历史运行版本 | 用途 |
|---|---|---|
| numpy / pandas / scipy | 2.4.4 / 3.0.5 / 1.17.1 | 数组、表格、优化 |
| scikit-learn | 1.9.0 | 编码、LR、树、RF、HGB |
| joblib / threadpoolctl | 1.5.3 / 3.6.0 | 串行保存与线程限制 |
| imbalanced-learn / sklearn-compat | 0.14.2 / 0.1.6 | 仅Balanced RF需要 |
| IPython / ipykernel | 9.17.1 / 7.3.0 | Notebook执行 |

普通实验可以使用主仓库的 `requirements.txt`。Balanced RF另需下列扩展依赖；
代码会在实际调用BRF前检查上述两个可选包自身的精确版本及其依赖约束；
缺包或版本不符会明确报错，不会换成普通RF，也不会自动安装或升级。


In [1]:
# Optional BRF dependencies: uncomment only when you intend to install into this kernel.
# %pip install imbalanced-learn==0.14.2 sklearn-compat==0.1.6


## 1. 配置、数据边界与共同定义

所有行号为0起点、右端不包含；保持CSV原顺序，不shuffle。

| 集合/评价窗 | 拟合范围 | 评价范围 |
|---|---|---|
| D1 | [0,12000) | [12000,16000) |
| D2 | [0,16000) | [16000,20000) |
| D3 | [0,20000) | [20000,24000) |
| D4 | [0,24000) | [24000,28831) |
| val | [0,28831) | [28831,35009) |
| test | 使用最终开发模型；不重训 | [35009,41188) |

recent_12000把相应拟合范围改为 `[max(0, stop−12000),stop)`；
早期校准另在每个训练前缀内部作80%/20%切分。其他例外在对应章节写明。
train/val的预处理只能在该折训练范围内fit；`duration`和`y`不进入特征。
这些前缀训练集有重叠，五窗不是五个独立样本；行数窗口也不等于等长日历时间。


In [2]:
from pathlib import Path
import gc, io, json, math, re, warnings, importlib.metadata
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.special import expit, logit
from scipy.optimize import minimize_scalar, brentq
import sklearn
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
from IPython.display import display, HTML

# Empty set: load definitions and parameter tables only. No model fitting or test read.
# Example: EXPERIMENTS = {"refinement", "window", "kalman"}
# To reproduce all development families: EXPERIMENTS = set(FAMILY_NAMES)
FAMILY_NAMES = [
    "diagnosis", "diagnosis_followup", "macro", "personal_macro", "weights",
    "early_calibration", "scope", "precision", "boosting", "binning",
    "imbalance", "history_scale", "history_diagnostic", "calibration",
    "refinement", "window", "kalman",
]
EXPERIMENTS = set()
FOLD_NAMES = ["D1", "D2", "D3", "D4", "val"]
# Use ["val"] for a bounded replay; previous-window thresholds then remain unavailable.
MODEL_FILTER = {}  # e.g. {"refinement": {"baseline", "drop_loans"}}
RUN_TEST = False   # Explicit opt-in, after development configurations are frozen.
DATA_PATH = Path("bank-additional-full.csv")
OUTPUT_DIR = Path("outputs/supplementary_reproduction")
# Generated outputs are not input dependencies and need not be committed.

FOLDS = [("D1", 12000, 16000), ("D2", 16000, 20000),
         ("D3", 20000, 24000), ("D4", 24000, 28831), ("val", 28831, 35009)]
VAL_BOUNDS = np.array([28831, 29861, 30891, 31921, 32951, 33980, 35009])
assert set(EXPERIMENTS) <= set(FAMILY_NAMES)
assert set(FOLD_NAMES) <= {f[0] for f in FOLDS}
assert DATA_PATH.is_file(), "Place the original CSV beside this Notebook."

def read_development(path):
    # An unbuffered bounded read avoids a parser reading ahead into the test suffix.
    with Path(path).open("rb", buffering=0) as handle:
        lines = [handle.readline()]
        for _ in range(35009):
            line = handle.readline()
            assert line.endswith(b"\n")
            lines.append(line)
        offset = handle.tell()
    frame = pd.read_csv(io.BytesIO(b"".join(lines)), sep=";")
    assert frame.shape == (35009, 21)
    assert frame.iloc[:28831].y.eq("yes").sum() == 1606
    assert frame.iloc[28831:].y.eq("yes").sum() == 658
    return frame, offset

DEV, DEV_BYTE_STOP = read_development(DATA_PATH)
X_DEV = DEV.drop(columns=["y", "duration"])
Y_DEV = DEV.y.eq("yes").to_numpy(dtype=int)
TEST_STARTED = False
ARTIFACTS, FIT_CACHE, PRED_CACHE, FINAL_MODELS = {}, {}, {}, {}
METRICS, GROUP_METRICS, RELIABILITY, FITS, PARAMETERS, THRESHOLD_LOG = [], [], [], [], [], []
CONTRASTS, AUX_GROUPS, COEFFICIENTS, TOPK_DIAGNOSTICS, TOPK_COMPOSITION = [], [], [], [], []
THRESHOLD_HISTORY = {}
print("Development rows:", len(DEV), "| Test rows read: 0 | Selected families:", sorted(EXPERIMENTS))

def display_table(frame, first=10):
    # Keep every row/column in the portable output, with a compact initial view.
    front=frame.iloc[:first].to_html(index=False)
    rest=frame.iloc[first:]
    if len(rest):
        front+=f"<details><summary>展开其余 {len(rest)} 行</summary>"+rest.to_html(index=False)+"</details>"
    display(HTML(front))


Development rows: 35009 | Test rows read: 0 | Selected families: []


### 1.1 共同指标与操作点

记 \(y_i\in\{0,1\}\)，概率 \(p_i\)，阈值预测 \(\hat y_i=\mathbf1[p_i\ge t]\)。
\[
\mathrm{Precision}=\frac{TP}{TP+FP},\quad
\mathrm{Recall}=\frac{TP}{TP+FN},\quad
F_1=\frac{2TP}{2TP+FP+FN},\quad
\mathrm{Accuracy}=\frac{TP+TN}{N}.
\]
\[
\mathrm{Brier}=\frac1N\sum_i(p_i-y_i)^2,\qquad
\mathrm{LogLoss}=-\frac1N\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)].
\]
log loss将概率裁剪至float64的 \([\epsilon,1-\epsilon]\)。
ECE使用10个等宽概率箱：
\[
\mathrm{ECE}_{10}=\sum_{b=1}^{10}\frac{|B_b|}{N}
\left|\overline p_{B_b}-\overline y_{B_b}\right|.
\]
空箱贡献为0。Specificity为 \(TN/(TN+FP)\)，Balanced Accuracy为
\((\mathrm{Recall}+\mathrm{Specificity})/2\)，Lift为Precision除以该评价样本正类率。
无预测正例时Precision未定义，保存为NaN；F1的零分母返回0。
ROC-AUC是正例得分高于负例的配对概率，平分给0.5；
AP按得分阈值形成的召回增量加权Precision，使用 `average_precision_score`。

**Top5/10/20%：** 按概率降序、原始行号升序打破同分，恰好联系 \(\lceil qN\rceil\) 人。
这是联系预算规则，不是用test标签选阈值。分组表先确定全体名单，再按poutcome切组。

**F1阈值：** 只用指定开发/校准窗最大化F1，数值并列容差 \(10^{-12}\)，取较高阈值；
历史分箱实验使用严格数值相等判并列，代码保留这个区别。
**Precision约束：** 在完整同分组边界里，要求Precision≥目标且入选人数≥最小支持数；
依次最大化TP、Precision、阈值。不可达时记录None，不能回退0.5冒充达标。
本窗选阈值并在本窗评分标为 `SELECTION_ONLY`；下一窗使用前一窗已选阈值才叫前向迁移。


In [3]:
def top_mask(p, row_ids, fraction):
    p, row_ids = np.asarray(p), np.asarray(row_ids)
    selected = np.zeros(len(p), dtype=bool)
    selected[np.lexsort((row_ids, -p))[:math.ceil(len(p)*fraction)]] = True
    return selected

def choose_f1_threshold(y, p, exact_ties=False):
    precision, recall, thresholds = precision_recall_curve(y, p, drop_intermediate=False)
    score = np.divide(2*precision[:-1]*recall[:-1], precision[:-1]+recall[:-1],
                      out=np.zeros_like(thresholds), where=precision[:-1]+recall[:-1]>0)
    best = score == score.max() if exact_ties else np.isclose(score, score.max(), rtol=0, atol=1e-12)
    return float(thresholds[np.flatnonzero(best)[-1]])

def choose_precision_threshold(y, p, target, minimum_n=30):
    order = np.argsort(-np.asarray(p), kind="stable")
    ps, ys = np.asarray(p)[order], np.asarray(y)[order]
    ends = np.r_[np.flatnonzero(ps[:-1] != ps[1:]), len(ps)-1]
    count = ends+1
    tp = np.cumsum(ys)[ends]
    precision = tp/count
    feasible = np.flatnonzero((precision >= target) & (count >= minimum_n))
    if not len(feasible):
        return None
    j = max(feasible, key=lambda k: (tp[k], precision[k], ps[ends[k]]))
    return float(ps[ends[j]])

def choose_recall_threshold(y, p, target):
    order = np.argsort(-np.asarray(p), kind="stable")
    ps, ys = np.asarray(p)[order], np.asarray(y)[order]
    ends = np.r_[np.flatnonzero(ps[:-1] != ps[1:]), len(ps)-1]
    j = np.flatnonzero(np.cumsum(ys)[ends] >= target*np.sum(ys))[0]
    return float(ps[ends[j]])

def probability_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p, dtype=float)
    assert len(y) and np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    eps = np.finfo(float).eps
    q = np.clip(p, eps, 1-eps)
    bins = np.minimum((p*10).astype(int), 9)
    ece = sum(abs(float(p[bins==j].sum()-y[bins==j].sum()))/len(y) for j in range(10))
    both = len(np.unique(y)) == 2
    return dict(n=len(y), yes=int(y.sum()), positive_rate=float(y.mean()),
                mean_probability=float(p.mean()), brier=float(np.mean((p-y)**2)),
                log_loss=float(-np.mean(y*np.log(q)+(1-y)*np.log1p(-q))), ece=ece,
                roc_auc=float(roc_auc_score(y,p)) if both else np.nan,
                average_precision=float(average_precision_score(y,p)) if y.sum() else np.nan)

def classification_metrics(y, selected):
    y, selected = np.asarray(y), np.asarray(selected, dtype=bool)
    tp = int(y[selected].sum()); fp = int(selected.sum())-tp
    fn = int(y.sum())-tp; tn = len(y)-tp-fp-fn
    precision = tp/(tp+fp) if tp+fp else np.nan
    recall = tp/(tp+fn) if tp+fn else np.nan
    specificity = tn/(tn+fp) if tn+fp else np.nan
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, selected_n=tp+fp,
                accuracy=(tp+tn)/len(y), precision=precision, precision_defined=bool(tp+fp),
                recall=recall, specificity=specificity, balanced_accuracy=(recall+specificity)/2,
                f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,
                lift=precision/y.mean() if y.sum() else np.nan)

def evaluate_rules(y, p, row_ids, extra_rules=None):
    rules = {"fixed_0.5": .5, "top5": None, "top10": None, "top20": None}
    rules.update(extra_rules or {})
    common = probability_metrics(y,p)
    out, masks = [], {}
    for rule, threshold in rules.items():
        if rule.startswith("top") and rule in {"top5","top10","top20"}:
            selected = top_mask(p,row_ids,int(rule[3:])/100)
        elif threshold is None:
            continue  # Unreachable constraints are recorded separately, never replaced by .5.
        else:
            selected = np.asarray(p) >= threshold
        masks[rule] = selected
        out.append(dict(rule=rule, threshold=threshold, **common, **classification_metrics(y,selected)))
    return out, masks

def append_results(family, name, fold, ids, p, extra_rules=None, data=None):
    data = DEV.loc[ids] if data is None else data
    y = data.y.eq("yes").to_numpy(dtype=int)
    rows, masks = evaluate_rules(y,p,ids,extra_rules)
    meta = dict(family=family,model=name,fold=fold,split="test" if fold=="test" else "development")
    METRICS.extend(dict(**meta,**r) for r in rows)
    # Decide on the whole window first, then slice; no within-group reranking.
    for group in ["nonexistent","failure","success"]:
        mask = data.poutcome.eq(group).to_numpy()
        if not mask.any():
            continue
        prob = probability_metrics(y[mask],np.asarray(p)[mask])
        for r in rows:
            GROUP_METRICS.append(dict(**meta,group=group,rule=r["rule"],threshold=r["threshold"],
                 **prob,**classification_metrics(y[mask],masks[r["rule"]][mask])))
    bins = np.minimum((np.asarray(p)*10).astype(int),9)
    for j in range(10):
        mask = bins==j
        RELIABILITY.append(dict(**meta,bin=j,n=int(mask.sum()),
            mean_probability=float(np.asarray(p)[mask].mean()) if mask.any() else np.nan,
            observed_rate=float(y[mask].mean()) if mask.any() else np.nan))

def show_family(family, fold="val"):
    table = pd.DataFrame(METRICS)
    if table.empty or not table.family.eq(family).any():
        print(f"{family}: NOT_RUN (enable it in EXPERIMENTS).")
        return
    chosen = table[table.family.eq(family)&table.fold.eq(fold)&table.rule.eq("top10")]
    if chosen.empty:
        chosen = table[table.family.eq(family)&table.rule.eq("top10")]
    display_table(chosen)
    print("All calculated rows remain available in METRICS and the generated metrics.csv.")


### 1.2 特征表示与LR目标

类别列使用完整one-hot（`drop=None`），未见类别忽略；数值标准化只使用训练统计量：
\[
z=(x-\mu_{\mathrm{train}})/s_{\mathrm{train}}.
\]
历史表示为
\[
\mathrm{campaign\_log}=\log(1+\mathrm{campaign}),\quad
\mathrm{previous\_group}\in\{0,1,2+\},
\]
\[
\mathrm{pdays\_known}=\mathbf1[\mathrm{pdays}\ne999],\qquad
\mathrm{pdays\_log}=\begin{cases}\log(1+\mathrm{pdays}),&\mathrm{pdays}\ne999\\0,&\mathrm{pdays}=999.\end{cases}
\]
999是单独编码的特殊值；这里不把它解释成已经证实的业务状态。

LR为 \(p_i=\sigma(b+\beta^\top x_i)\)，L2/`lbfgs`。与实现等价的目标可写为
\[
\min_{\beta,b}\ \sum_i w_i\log(1+\exp[-(2y_i-1)(b+\beta^\top x_i)])
   +\frac{\|\beta\|_2^2}{2C},
\]
截距不惩罚。除单独注明外，`C=1,tol=1e-7,max_iter=20000`，无类权重。
改变总权重会改变相对正则强度，因此采样和类别权重实验保留了原权重归一化。

**两种年龄样条不可混用。** 早期diagnosis：5个训练分位数结点、2次样条、
`extrapolation="linear"`，之后再StandardScaler。后续history表示：4个训练分位数结点、
2次样条、`extrapolation="constant"`，不再标准化样条列；两者都`include_bias=False`。


In [4]:
A_CATEGORICAL = ["job","marital","education","default","housing","loan","contact","month","day_of_week","poutcome"]
A_NUMERIC = ["age","campaign","pdays","previous","emp.var.rate","cons.price.idx","cons.conf.idx","euribor3m","nr.employed"]
MACRO = ["emp.var.rate","cons.price.idx","cons.conf.idx","euribor3m","nr.employed"]
PERSONAL = ["job","marital","education","default","housing","loan"]

def logistic(C=1., tol=1e-7, max_iter=20000):
    options = dict(C=C,solver="lbfgs",tol=tol,max_iter=max_iter,fit_intercept=True,class_weight=None)
    options.update({"l1_ratio":0.0} if tuple(map(int,sklearn.__version__.split(".")[:2])) >= (1,8)
                   else {"penalty":"l2"})
    return LogisticRegression(**options)

class HistoryFeatures(TransformerMixin, BaseEstimator):
    def fit(self,X,y=None):
        assert "y" not in X and "duration" not in X
        self.feature_names_in_ = np.asarray(X.columns,dtype=object)
        return self
    def transform(self,X):
        z = X.copy()
        z["campaign_log"] = np.log1p(z.campaign)
        z["previous_group"] = np.where(z.previous==0,"0",np.where(z.previous==1,"1","2plus"))
        z["pdays_known"] = (z.pdays!=999).astype(float)
        z["pdays_log"] = np.where(z.pdays!=999,np.log1p(z.pdays),0.)
        return z

class BinnedHistory(HistoryFeatures):
    def transform(self,X):
        z = super().transform(X)
        z["campaign_bin"] = np.where(z.campaign>=4,"4plus",z.campaign.astype(str))
        z["pdays_bin"] = np.select(
            [z.pdays.eq(999),z.pdays.eq(0),z.pdays.between(1,7),z.pdays.between(8,30)],
            ["sentinel_999","0_days","1_7_days","8_30_days"],default="over_30_days")
        return z

class AgeJobInteraction(TransformerMixin,BaseEstimator):
    def fit(self,X,y=None): return self
    def transform(self,X):
        return np.column_stack([(X.age.to_numpy()-25)/10*X.job.eq("student").to_numpy(),
                                (X.age.to_numpy()-60)/10*X.job.eq("retired").to_numpy()])
    def get_feature_names_out(self,input_features=None):
        return np.array(["student_age_decades_from25","retired_age_decades_from60"],dtype=object)

def history_preprocessor(spec):
    drop = set(spec.get("drop",[]))
    hist = spec.get("history",3)
    cat = [c for c in PERSONAL+["contact","month","day_of_week","poutcome"] if c not in drop]
    if hist>=1 and "previous" not in drop: cat += ["previous_group"]
    num = ["campaign_log"] if "campaign" not in drop else []
    if hist>=3 and "pdays" not in drop: num += ["pdays_log"]
    binary = ["pdays_known"] if hist>=2 and "pdays" not in drop else []
    cb, pb = spec.get("campaign_bin",False), spec.get("pdays_bin",False)
    if cb:
        cat += ["campaign_bin"]; num = [c for c in num if c!="campaign_log"]
    if pb:
        cat += ["pdays_bin"]; num = [c for c in num if c!="pdays_log"]; binary = []
    dense = spec.get("dense",False)
    groups = [("cat",OneHotEncoder(handle_unknown="ignore",drop=None,sparse_output=not dense),cat)]
    if num: groups.append(("num",StandardScaler(),num))
    if binary: groups.append(("binary","passthrough",binary))
    if "age" not in drop:
        tr = StandardScaler() if spec.get("age","spline")=="linear" else SplineTransformer(
            n_knots=4,degree=2,knots="quantile",extrapolation="constant",include_bias=False)
        groups.append(("age",tr,["age"]))
    penalty = spec.get("macro_penalty")
    if penalty: groups.append(("macro",StandardScaler(),MACRO))
    if spec.get("interaction"):
        groups.append(("interaction",AgeJobInteraction(),["age","job"]))
    pre = ColumnTransformer(groups,transformer_weights={"macro":1/np.sqrt(penalty)} if penalty else None,
                            sparse_threshold=0. if dense else 1.)
    # The personal-only design does not require unused contact fields at prediction.
    features = "passthrough" if spec.get("personal_only") else BinnedHistory() if cb or pb else HistoryFeatures()
    return Pipeline([("features",features),("preprocess",pre)])

def diagnosis_preprocessor(spec):
    numeric = [c for c in A_NUMERIC if not(spec.get("log_counts") and c in ["campaign","previous"])
               and not(spec.get("spline") and c=="age")]
    groups = [("cat",OneHotEncoder(handle_unknown="ignore",drop=None),A_CATEGORICAL),
              ("num",StandardScaler(),numeric)]
    if spec.get("log_counts"):
        groups.append(("log_counts",Pipeline([
            ("log",FunctionTransformer(np.log1p,feature_names_out="one-to-one")),
            ("scale",StandardScaler())]),["campaign","previous"]))
    if spec.get("spline"):
        groups.append(("age_spline",Pipeline([
            ("spline",SplineTransformer(n_knots=5,degree=2,knots="quantile",
                                        include_bias=False,extrapolation="linear")),
            ("scale",StandardScaler())]),["age"]))
    return Pipeline([("preprocess",ColumnTransformer(groups,sparse_threshold=1.))])

def make_preprocessor(spec):
    if spec.get("representation","history")=="diagnosis":
        return diagnosis_preprocessor(spec)
    if spec.get("representation")=="macro":
        numeric = [c for c in A_NUMERIC if c not in spec.get("drop_macro",[])]
        return Pipeline([("preprocess",ColumnTransformer([
            ("cat",OneHotEncoder(handle_unknown="ignore",drop=None),A_CATEGORICAL),
            ("num",StandardScaler(),numeric)],sparse_threshold=1.))])
    return history_preprocessor(spec)


### 1.3 训练与采样实现

预处理始终在原训练样本上fit，然后对编码后的训练矩阵重采样；
不会对val/test采样，也不会把复制后的样本分布用于重新学习编码或标准化参数。
每次只保留当前模型在内存中，训练完保存到本次运行目录；相同配置与训练边界复用同一次拟合。
模型文件是运行时生成的，不要求从其他机器复制。

Balanced RF历史设置明确为：
`n_estimators=300, criterion="gini", min_samples_leaf=5, max_features="sqrt",
sampling_strategy="all", replacement=True, bootstrap=False, class_weight=None, n_jobs=1`。
普通/加权RF使用`bootstrap=True`；加权RF是`class_weight="balanced_subsample"`。
下面保留实际实现，不用“平衡采样”的名称掩盖不同的bootstrap含义。

**树与森林的概率计算：** 对落入叶节点 \(L_b(x)\) 的样本，树b输出该叶内的加权正类比例；
森林对300棵树的概率取平均：
\[
p_b(x)=\frac{\sum_{i\in L_b(x)}w_{ib}y_i}{\sum_{i\in L_b(x)}w_{ib}},\qquad
p_{\rm RF}(x)=\frac1{300}\sum_{b=1}^{300}p_b(x).
\]
这里的求和按该树实际训练样本及其抽样次数计数。普通树/RF使用单位类别权重；
加权RF在各bootstrap样本内采用 \(w_{c,b}=n_b/(2n_{c,b})\)；
BRF先在每棵树内部均衡采样，再以单位类别权重训练。二分类节点的Gini不纯度为
\[
G(L)=1-\pi_L^2-(1-\pi_L)^2.
\]
分裂以加权子节点不纯度下降为准，受深度、最小叶样本数和随机特征选择约束。


In [5]:
def sampling_indices(y,strategy="none",seed=0):
    n = len(y); neg = np.flatnonzero(y==0); pos = np.flatnonzero(y==1)
    assert len(neg)>len(pos)>0
    if strategy in ["none","balanced"]:
        index = np.arange(n)
        weight = np.ones(n) if strategy=="none" else np.where(y==1,n/(2*len(pos)),n/(2*len(neg)))
    else:
        rng = np.random.default_rng(seed); factor = 2 if strategy.endswith("1to2") else 1
        if strategy.startswith("ros"):
            needed = int(np.ceil(len(neg)/factor))
            index = np.r_[np.arange(n),rng.choice(pos,size=needed-len(pos),replace=True)]
        elif strategy.startswith("rus"):
            index = np.sort(np.r_[pos,rng.choice(neg,size=factor*len(pos),replace=False)])
        else:
            raise ValueError(strategy)
        weight = np.full(len(index),n/len(index))
    assert np.isclose(weight.sum(),n)
    return index,weight

def check_brf_dependency():
    # Optional dependency is only required when BRF is selected.
    try:
        from imblearn.ensemble import BalancedRandomForestClassifier
        from packaging.requirements import Requirement
        from packaging.version import Version
    except ImportError as exc:
        raise ImportError("BRF requires imbalanced-learn==0.14.2 and sklearn-compat==0.1.6; "
                          "install the declared extras in the Notebook kernel environment.") from exc
    for package,expected in {"imbalanced-learn":"0.14.2","sklearn-compat":"0.1.6"}.items():
        actual=importlib.metadata.version(package)
        if actual!=expected:
            raise RuntimeError(f"Historical BRF replay requires {package}=={expected}; found {actual}.")
    for package in ["imbalanced-learn","sklearn-compat"]:
        for item in importlib.metadata.requires(package) or []:
            r = Requirement(item)
            if r.marker and not r.marker.evaluate(): continue
            assert Version(importlib.metadata.version(r.name)) in r.specifier, item
    return BalancedRandomForestClassifier

def make_classifier(spec):
    kind = spec.get("kind","LR")
    if kind=="LR":
        return logistic(spec.get("C",1.),spec.get("tol",1e-7),spec.get("max_iter",20000))
    if kind=="TREE":
        return DecisionTreeClassifier(max_depth=spec["depth"],min_samples_leaf=spec["leaf"],
                                      random_state=42,criterion="gini")
    if kind=="HGB":
        return HistGradientBoostingClassifier(
            loss="log_loss",max_bins=255,max_features=1.0,
            max_depth=spec["depth"],max_leaf_nodes=spec.get("max_leaf_nodes",2**spec["depth"]),
            min_samples_leaf=spec["leaf"],l2_regularization=spec["l2"],
            learning_rate=spec["learning_rate"],max_iter=spec["iterations"],
            early_stopping=False,random_state=42,class_weight=None,categorical_features=None)
    common = dict(n_estimators=300,criterion="gini",max_depth=None,min_samples_split=2,
                  min_samples_leaf=5,max_features="sqrt",n_jobs=1,random_state=spec["seed"])
    if kind=="BRF":
        return check_brf_dependency()(**common,sampling_strategy="all",replacement=True,
                                      bootstrap=False,class_weight=None)
    if kind=="RF":
        return RandomForestClassifier(**common,bootstrap=True,
            class_weight="balanced_subsample" if spec.get("weighted_rf") else None)
    raise ValueError(kind)

def safe_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+","_",text)

def prediction_from_bundle(bundle,X,decision=False):
    if bundle.get("constant") is not None:
        p = np.full(len(X),bundle["constant"])
        return logit(np.clip(p,np.finfo(float).eps,1-np.finfo(float).eps)) if decision else p
    a = bundle["preprocessor"].transform(X)
    if bundle.get("scale") is not None:
        a = a.multiply(bundle["scale"]).tocsr() if sparse.issparse(a) else a*bundle["scale"]
    clf = bundle["classifier"]
    return clf.decision_function(a) if decision else clf.predict_proba(a)[:,1]

def fit_key(spec,start,stop):
    # Window length has already been resolved into [start, stop).
    effective = {k:v for k,v in spec.items() if k!="recent"}
    return json.dumps([effective,start,stop],sort_keys=True)

def fit_bundle(spec,start,stop):
    assert not TEST_STARTED, "All fitting must finish before test evaluation begins."
    assert 0<=start<stop<=28831
    key = fit_key(spec,start,stop)
    if key in FIT_CACHE:
        return FIT_CACHE[key]
    X, y = X_DEV.iloc[start:stop], Y_DEV[start:stop]
    if spec.get("constant"):
        bundle = dict(constant=float(y.mean()))
        fit_info = dict(iterations=0,converged=True,features=0,weight_sum=len(y),fitted=False)
    else:
        pre = make_preprocessor(spec)
        a = pre.fit_transform(X)
        scale = None
        if "poutcome_scale" in spec:
            names = pre.named_steps["preprocess"].get_feature_names_out()
            scale = np.where(np.char.startswith(names.astype(str),"cat__poutcome_"),spec["poutcome_scale"],1.)
            a = a.multiply(scale).tocsr() if sparse.issparse(a) else a*scale
        index, weight = np.arange(len(y)), None
        if "sampling" in spec:
            index,weight = sampling_indices(y,spec["sampling"],spec.get("seed",0))
        if "yes_weight" in spec:
            ratio = float((y==0).sum()/(y==1).sum()) if spec["yes_weight"]=="balanced" else float(spec["yes_weight"])
            weight = np.where(y==1,ratio,1.); weight /= weight.mean()
        if spec.get("time_weighted"):
            weight = np.exp2(-(len(y)-1-np.arange(len(y)))/len(y)); weight /= weight.mean()
        clf = make_classifier(spec)
        with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            if weight is None: clf.fit(a[index],y[index])
            else: clf.fit(a[index],y[index],sample_weight=weight)
        converged = not any(issubclass(w.category,ConvergenceWarning) for w in caught)
        if spec.get("tol",1e-7)<=1e-7:
            assert converged, "Strict historical configuration did not converge."
        iterations = int(np.max(np.atleast_1d(getattr(clf,"n_iter_",0))))
        fit_info = dict(iterations=iterations,converged=converged,features=a.shape[1],
                        weight_sum=float(np.sum(weight)) if weight is not None else len(y),fitted=True,
                        warnings=[str(w.message) for w in caught])
        bundle = dict(preprocessor=pre,classifier=clf,scale=scale)
        del a
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    artifact_id = f"fit_{len(ARTIFACTS):04d}"
    path = OUTPUT_DIR/(artifact_id+".joblib")
    joblib.dump(bundle,path,compress=3)
    ARTIFACTS[artifact_id] = dict(path=str(path),spec=spec,train_start=start,train_stop=stop)
    FIT_CACHE[key] = artifact_id
    FITS.append(dict(artifact=artifact_id,train_start=start,train_stop=stop,**fit_info,spec=json.dumps(spec,sort_keys=True)))
    del bundle
    gc.collect()
    return artifact_id

def predict_artifact(artifact,ids,decision=False):
    ids = np.asarray(ids)
    key = (artifact,int(ids[0]),int(ids[-1])+1,decision)
    if key not in PRED_CACHE:
        bundle = joblib.load(ARTIFACTS[artifact]["path"])
        with threadpool_limits(limits=1):
            PRED_CACHE[key] = prediction_from_bundle(bundle,X_DEV.loc[ids],decision)
        del bundle;gc.collect()
    return PRED_CACHE[key].copy()

def register_final(family,name,artifact,rules=None,**mapping):
    # This contains development-derived parameters only. It is frozen before test.
    FINAL_MODELS[(family,name)] = dict(artifact=artifact,rules=dict(rules or {}),**mapping)


### 1.4 完整配置目录

以下配置由既有实验网格逐项整理。参数表中的空白表示该参数不适用于该配置；共同默认值见前文。


In [6]:
def history_spec(**kw):
    return dict(age="spline",history=3,macro_penalty=None,drop=[],**kw)

CONTACT = dict(age="spline",history=3,macro_penalty=None,drop=["month","day_of_week"],C=.01)
PERSONAL_SCOPE = dict(age="spline",history=0,macro_penalty=None,
                     drop=["contact","month","day_of_week","poutcome","previous","pdays","campaign"],
                     personal_only=True)
CONTACT_SCOPE = {**CONTACT,"C":1.}

def personal_candidates():
    result = {}
    for name,age,hist in [("P0_linear","linear",0),("P1_age_spline","spline",0),
                          ("P2_previous","spline",1),("P3_pdays_flag","spline",2),("P_full","spline",3)]:
        result[name] = dict(age=age,history=hist,macro_penalty=None,drop=[])
    for penalty in [1,10,100]:
        result[f"M_penalty{penalty}"] = {**result["P_full"],"macro_penalty":penalty}
    for arm,base in [("P","P_full"),("M","M_penalty10")]:
        for label,cols in [("age",["age"]),("job",["job"]),("education",["education"]),
            ("marital",["marital"]),("default",["default"]),("loans",["housing","loan"]),
            ("contact",["contact"]),("month",["month"]),("weekday",["day_of_week"]),
            ("history",["poutcome","previous","pdays"]),("campaign",["campaign"])]:
            result[f"{arm}_drop_{label}"] = {**result[base],"drop":cols}
    result["train_prior"] = {"constant":True}
    return result

CONFIGS = {}
CONFIGS["diagnosis"] = {
    name:dict(representation="diagnosis",tol=1e-4,max_iter=5000,**s)
    for name,s in {"baseline":{"C":1},"C_0.1":{"C":.1},"C_10":{"C":10},
                  "log_counts":{"C":1,"log_counts":True},"age_spline":{"C":1,"spline":True},
                  "log_counts_age_spline":{"C":1,"log_counts":True,"spline":True},
                  "recent_12000":{"C":1,"recent":12000}}.items()}
CONFIGS["diagnosis"]["train_prior_dummy"] = {"constant":True}
CONFIGS["diagnosis_followup"] = {"age_spline_strict":dict(representation="diagnosis",C=1,spline=True)}
CONFIGS["macro"] = {name:dict(representation="macro",drop_macro=drop) for name,drop in
                   {"M1_all_macro":[],"M2_no_macro":MACRO,"M3_three_macro":["emp.var.rate","nr.employed"]}.items()}
CONFIGS["macro"]["train_prior_dummy"] = {"constant":True}
CONFIGS["personal_macro"] = personal_candidates()
CONFIGS["weights"] = {
    f"{arm}_w{w}_i{int(interaction)}":
    dict(age="spline",history=3,macro_penalty=None if arm=="P" else 10,drop=[],
         yes_weight=w,interaction=interaction)
    for arm in ["P","M10"] for w in ["1","2","5","balanced"] for interaction in [False,True]}
CONFIGS["scope"] = {"personal_only":PERSONAL_SCOPE,"personal_contact_history":CONTACT_SCOPE}
CONFIGS["precision"] = dict(CONFIGS["scope"])
for scope in ["personal","contact"]:
    base = PERSONAL_SCOPE if scope=="personal" else CONTACT_SCOPE
    for age in ["linear","spline"]:
        for C in [.01,.1,1,10]:
            CONFIGS["precision"][f"{scope}_LR_{age}_C{C}"] = {**base,"age":age,"C":C}
    for depth in [2,3,4]:
        for leaf in [100,300]:
            CONFIGS["precision"][f"{scope}_TREE_d{depth}_leaf{leaf}"] = {
                **base,"age":"linear","kind":"TREE","depth":depth,"leaf":leaf}
CONFIGS["boosting"] = {
    f"{scope}_HGB_d{depth}_l2{l2}":dict(
        **{**(PERSONAL_SCOPE if scope=="personal" else CONTACT_SCOPE),"age":"linear"},
        dense=True,kind="HGB",depth=depth,leaf=100,l2=l2,learning_rate=.05,iterations=150)
    for scope in ["personal","contact"] for depth in [2,3] for l2 in [1,10]}
CONFIGS["binning"] = {
    f"{arm}__{variant}":dict(age="spline",history=3,macro_penalty=penalty,drop=[],
                           campaign_bin=cb,pdays_bin=pb)
    for arm,penalty in [("P_full",None),("M_penalty10",10)]
    for variant,cb,pb in [("current",False,False),("campaign_only",True,False),
                          ("pdays_only",False,True),("both",True,True)]}
IMBALANCE_BASES = {
    "contact_LR_spline_C0.01":CONTACT,
    "M_drop_weekday":CONFIGS["personal_macro"]["M_drop_weekday"],
    "P_drop_weekday":CONFIGS["personal_macro"]["P_drop_weekday"],
    "M2_no_macro":CONFIGS["macro"]["M2_no_macro"],
    "diagnosis_age_spline":dict(representation="diagnosis",C=1,spline=True),
}
CONFIGS["imbalance"] = {}
for name,spec in IMBALANCE_BASES.items():
    for strategy,seeds in [("none",[0]),("balanced",[0]),("ros_1to2",[17,42,2026]),
                            ("ros_1to1",[17,42,2026]),("rus_1to2",[17,42,2026]),
                            ("rus_1to1",[17,42,2026])]:
        for seed in seeds:
            CONFIGS["imbalance"][f"{name} | {strategy} | seed={seed}"] = {
                **spec,"sampling":strategy,"seed":seed}
for kind,strategy in [("RF","ordinary_rf"),("RF","weighted_rf"),("BRF","balanced_rf")]:
    for seed in [17,42,2026]:
        CONFIGS["imbalance"][f"contact_LR_spline_C0.01 | {strategy} | seed={seed}"] = {
            **CONTACT,"kind":kind,"seed":seed,"weighted_rf":strategy=="weighted_rf"}
assert len(CONFIGS["imbalance"])==79
DIAGNOSTIC_BASES = {
    "LR_N":{**CONTACT,"sampling":"none","seed":0},
    "LR_W":{**CONTACT,"sampling":"balanced","seed":0},
    "LR_O2":{**CONTACT,"sampling":"ros_1to2","seed":42},
    "RF":{**CONTACT,"kind":"RF","seed":42,"weighted_rf":False},
}
DIAGNOSTIC_BASES.update({name:{**IMBALANCE_BASES[name],"sampling":"none","seed":0}
                       for name in ["M_drop_weekday","P_drop_weekday","M2_no_macro","diagnosis_age_spline"]})
CONFIGS["history_diagnostic"] = DIAGNOSTIC_BASES
CONFIGS["history_scale"] = {f"poutcome_scale{s}":{**CONTACT,"sampling":"none","seed":0,"poutcome_scale":s}
                            for s in [.5,1.,2.]}
CONFIGS["refinement"] = {
    "baseline":CONTACT,
    "pdays_coarse":{**CONTACT,"pdays_bin":True},
    "drop_loans":{**CONTACT,"drop":["month","day_of_week","housing","loan"]},
    "time_weighted":{**CONTACT,"time_weighted":True},
}
CONFIGS["window"] = {}
for kind in ["RF","HGB"]:
    for history in ["full_history","recent_12000"]:
        spec = {**CONTACT,"kind":kind,"dense":True}
        if history=="recent_12000": spec["recent"]=12000
        if kind=="RF": spec.update(seed=42,weighted_rf=False)
        else: spec.update(depth=2,max_leaf_nodes=31,leaf=20,l2=1,learning_rate=.1,iterations=100)
        CONFIGS["window"][f"{kind} | {history}"] = spec

# These inherited LR options are not applicable to tree estimators.
for configs in CONFIGS.values():
    for spec in configs.values():
        if spec.get("kind","LR")!="LR":
            for key in ["C","tol","max_iter"]:
                spec.pop(key,None)
        if spec.get("kind")=="HGB":
            spec.update(loss="log_loss",max_bins=255,max_features=1.0)


### 1.5 公共执行器

各章节调用这个执行器；它实际进行训练、预测、阈值选择、完整指标和分组统计，不读取历史成绩填表。


In [7]:
def subgroup_history_diagnostics():
    """Data diagnostics underlying the calibration/feature-scale pilot; development only."""
    records=[]
    for split,start,end in [("train",0,28831),("val",28831,35009)]:
        part=DEV.iloc[start:end]
        for field in ["poutcome","previous","pdays"]:
            groups=part.assign(target=part.y.eq("yes").astype(int)).groupby(field,dropna=False).target
            for level,group in groups:
                records.append(dict(split=split,field=field,level=level,n=len(group),
                                    yes=int(group.sum()),positive_rate=float(group.mean())))
    return pd.DataFrame(records)

def top_k_tied_diagnostic(y,p,row_ids,k):
    # Original HGB/LR error audit included every tie at the kth score.
    threshold=float(np.sort(p)[-k])
    selected=np.asarray(p)>=threshold
    return dict(requested_k=k,threshold=threshold,actual_n=int(selected.sum()),
                **probability_metrics(y,p),**classification_metrics(y,selected))

def append_group_slices(meta,frame,p,fields,rules):
    """Fixed whole-window thresholds, then subgroup slices; never select within groups."""
    y=frame.y.eq("yes").to_numpy(dtype=int)
    for field in fields:
        for level in frame[field].drop_duplicates():
            mask=frame[field].eq(level).to_numpy()
            common=probability_metrics(y[mask],p[mask])
            for rule,t in rules.items():
                if t is None:continue
                AUX_GROUPS.append(dict(**meta,field=field,level=str(level),rule=rule,threshold=t,
                    **common,**classification_metrics(y[mask],p[mask]>=t)))

def append_category_age_contrasts(name,fold,artifact,start,stop,ids):
    bundle=joblib.load(ARTIFACTS[artifact]["path"])
    pipe,clf=bundle["preprocessor"],bundle["classifier"]
    pre=pipe.named_steps["preprocess"];coef=clf.coef_[0]
    tx,vx=X_DEV.iloc[start:stop],X_DEV.loc[ids]
    transform=pipe.named_steps["features"]
    train,ev=transform.transform(tx),transform.transform(vx)
    refs={"job":"admin.","marital":"married","education":"high.school","default":"no",
          "housing":"no","loan":"no","contact":"telephone","month":"may","day_of_week":"mon",
          "poutcome":"nonexistent","previous_group":"0"}
    encoder=pre.named_transformers_["cat"]
    columns=next(cols for block,_,cols in pre.transformers_ if block=="cat")
    offset=pre.output_indices_["cat"].start
    for field,levels in zip(columns,encoder.categories_):
        ref=refs[field] if refs[field] in levels else str(levels[0])
        ri=list(levels).index(ref)
        for j,level in enumerate(levels):
            delta=float(coef[offset+j]-coef[offset+ri]);n=int(train[field].eq(level).sum())
            CONTRASTS.append(dict(family="personal_macro",fold=fold,model=name,field=field,
                level=str(level),reference=str(ref),delta_logit=delta,odds_ratio=float(np.exp(delta)),
                train_n=n,eval_n=int(ev[field].eq(level).sum()),
                train_reference_n=int(train[field].eq(ref).sum()),small_train_n=n<30))
        offset+=len(levels)
    if "age" in pre.named_transformers_:
        ages=[20,25,30,35,40,45,50,55,60,65,70,75,80]
        age=pre.named_transformers_["age"]
        difference=(age.transform(pd.DataFrame({"age":ages}))-
                    age.transform(pd.DataFrame({"age":[40]})))@coef[pre.output_indices_["age"]]
        for value,delta in zip(ages,difference):
            n=int(tx.age.eq(value).sum())
            CONTRASTS.append(dict(family="personal_macro",fold=fold,model=name,field="age",
                level=str(value),reference="40",delta_logit=float(delta),odds_ratio=float(np.exp(delta)),
                train_n=n,eval_n=int(vx.age.eq(value).sum()),small_train_n=n<30))
    # These are the descriptive train/evaluation slices in the original four-arm analysis.
    fields=["job","marital","education","default","housing","loan","contact","poutcome",
            "previous_group","age_band_5y","age_focus"]
    for split,raw,transformed in [("train",DEV.iloc[start:stop],train),("eval",DEV.loc[ids],ev)]:
        frame=transformed.copy();frame["y"]=raw.y
        frame["age_band_5y"]=pd.cut(frame.age,np.arange(15,101,5),right=False).astype(str)
        frame["age_focus"]=pd.cut(frame.age,[0,30,60,200],right=False,
                                   labels=["under30","30to59","60plus"]).astype(str)
        with threadpool_limits(limits=1):
            p=prediction_from_bundle(bundle,raw.drop(columns=["y","duration"]))
        append_group_slices(dict(family="personal_macro",model=name,fold=fold,split=split),
                            frame,p,fields,{"fixed_0.5":.5})
    del bundle

def append_topk_audit(family,name,fold,ids,p):
    frame=DEV.loc[ids].copy();y=Y_DEV[ids]
    frame["age_group"]=pd.cut(frame.age,[0,30,45,60,200],right=False).astype(str)
    frame["previous_group"]=np.where(frame.previous==0,"0",np.where(frame.previous==1,"1","2plus"))
    frame["pdays_known"]=np.where(frame.pdays==999,"no","yes")
    frame["campaign_group"]=pd.cut(frame.campaign,[0,1,3,5,np.inf]).astype(str)
    fields=["age_group","job","marital","education","default","housing","loan","contact",
            "previous_group","pdays_known","poutcome","campaign_group"]
    meta=dict(family=family,model=name,fold=fold,split="val")
    for k in [100,200,500]:
        row=top_k_tied_diagnostic(y,p,ids,k)
        TOPK_DIAGNOSTICS.append(dict(**meta,**row))
        selected=p>=row["threshold"]
        for field in fields:
            for level in frame[field].drop_duplicates():
                group=frame[field].eq(level).to_numpy();mask=selected&group
                tp=int(y[mask].sum());n=int(mask.sum());fp=n-tp
                TOPK_COMPOSITION.append(dict(**meta,top_k=k,field=field,level=str(level),
                    all_group_n=int(group.sum()),all_group_yes_rate=float(y[group].mean()),
                    selected_n=n,tp=tp,fp=fp,selected_precision=tp/n if n else np.nan,
                    share_of_TP=tp/row["tp"] if row["tp"] else np.nan,
                    share_of_FP=fp/row["fp"] if row["fp"] else np.nan))

def append_family_diagnostics(family,name,fold,artifact,start,stop,ids,p,extra):
    if family=="personal_macro" and name in {"P_full","M_penalty1","M_penalty10","M_penalty100"}:
        append_category_age_contrasts(name,fold,artifact,start,stop,ids)
    if family=="weights" and fold=="val":
        rules={"fixed_0.5":.5,"previous_window_TRANSFER_F1":extra.get("previous_window_TRANSFER_F1")}
        append_group_slices(dict(family=family,model=name,fold=fold,split="val"),
                            DEV.loc[ids],p,["job","education"],rules)
        bundle=joblib.load(ARTIFACTS[artifact]["path"])
        names=bundle["preprocessor"].named_steps["preprocess"].get_feature_names_out()
        COEFFICIENTS.extend(dict(family=family,model=name,fold=fold,feature=f,coefficient=float(b))
                            for f,b in zip(names,bundle["classifier"].coef_[0]))
        del bundle
    if fold=="val" and (family=="boosting" or (family=="precision" and name in {
        "personal_LR_spline_C0.01","contact_LR_linear_C0.01","contact_LR_spline_C0.01"})):
        append_topk_audit(family,name,fold,ids,p)

def show_family_diagnostics(family):
    for label,records in [("Category/age contrasts",CONTRASTS),("Historical subgroup slices",AUX_GROUPS),
                          ("Coefficient table",COEFFICIENTS),("Top-k with boundary ties",TOPK_DIAGNOSTICS),
                          ("Top-k TP/FP composition",TOPK_COMPOSITION)]:
        frame=pd.DataFrame(records)
        if not frame.empty and frame.family.eq(family).any():
            print(label);display_table(frame[frame.family.eq(family)])


In [8]:
def family_parameters(family):
    if family in CONFIGS:
        return pd.DataFrame([dict(model=n,**s) for n,s in CONFIGS[family].items()]).fillna("")
    return pd.DataFrame(PARAMETER_TABLES[family])

def threshold_rules(family,name,fold,y,p):
    extra,current = {},{}
    if family in {"weights","scope","refinement"}:
        current["F1"] = choose_f1_threshold(y,p)
    elif family=="binning":
        current["F1_exact_ties"] = choose_f1_threshold(y,p,exact_ties=True)
    elif family in {"precision","boosting"}:
        for target in [.3,.4,.5,.6]:
            for support in ([1,30] if family=="precision" else [30]):
                key=f"precision{target}_n{support}"
                current[key]=choose_precision_threshold(y,p,target,support)
    elif family=="imbalance":
        for target in [.2,.4,.6]:
            current[f"recall{target}"]=choose_recall_threshold(y,p,target)
    for key,t in current.items():
        rule="same_window_SELECTION_ONLY_"+key
        extra[rule]=t
        THRESHOLD_LOG.append(dict(family=family,model=name,fold=fold,rule=rule,threshold=t,
            threshold_source_fold=fold,target_recall=float(key[6:]) if key.startswith("recall") else None,
            status="UNREACHABLE" if t is None else "SELECTED_ON_DEVELOPMENT"))
    fi=[x[0] for x in FOLDS].index(fold)
    source=FOLDS[fi-1][0] if fi else None
    prior=THRESHOLD_HISTORY.get((family,name,source)) if source else None
    if family in {"weights","scope","refinement","precision","boosting","imbalance"}:
        for key in current:
            if not key.endswith("_n1"):
                rule=("previous_window_recall_TRANSFER_target"+key[6:] if family=="imbalance"
                      else "previous_window_TRANSFER_"+key)
                t=prior.get(key) if prior is not None else None
                extra[rule]=t
                THRESHOLD_LOG.append(dict(family=family,model=name,fold=fold,rule=rule,threshold=t,
                    threshold_source_fold=source,
                    target_recall=float(key[6:]) if key.startswith("recall") else None,
                    status=("NO_PREVIOUS_WINDOW" if prior is None else
                            "NO_PRIOR_THRESHOLD" if t is None else "APPLIED")))
    THRESHOLD_HISTORY[(family,name,fold)]=current
    return extra

def flush_results():
    if not FITS and not METRICS: return
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    for filename,records in [("metrics",METRICS),("subgroup_metrics",GROUP_METRICS),
                              ("reliability_bins",RELIABILITY),("fits",FITS),
                              ("parameters",PARAMETERS),("thresholds",THRESHOLD_LOG),
                              ("category_age_contrasts",CONTRASTS),("auxiliary_subgroups",AUX_GROUPS),
                              ("coefficients_val",COEFFICIENTS),("top_groups",TOPK_DIAGNOSTICS),
                              ("high_score_composition",TOPK_COMPOSITION)]:
        if records: pd.DataFrame(records).to_csv(OUTPUT_DIR/(filename+".csv"),index=False)

def run_standard(family):
    if family not in EXPERIMENTS:
        print(family,": NOT_RUN");return
    specs=CONFIGS[family]
    for fold,stop,end in FOLDS:
        if fold not in FOLD_NAMES:continue
        if family in {"binning","history_diagnostic","diagnosis_followup"} and fold!="val":continue
        for name,spec in specs.items():
            if family in MODEL_FILTER and name not in MODEL_FILTER[family]:continue
            start=max(0,stop-spec.get("recent",stop))
            artifact=fit_bundle(spec,start,stop)
            ids=np.arange(stop,end)
            p=predict_artifact(artifact,ids)
            extra=threshold_rules(family,name,fold,Y_DEV[ids],p)
            append_results(family,name,fold,ids,p,extra)
            append_family_diagnostics(family,name,fold,artifact,start,stop,ids,p,extra)
            if fold=="val":
                prefix="saved_dev_" if family=="imbalance" else "saved_development_"
                register_final(family,name,artifact,{prefix+k:v for k,v in extra.items()})
        print(family,fold,"complete")
        flush_results()
    # The strict follow-up also fitted a D4 baseline; it was not an extra final-val model.
    if family=="diagnosis_followup" and "D4" in FOLD_NAMES:
        a=fit_bundle(dict(representation="diagnosis",C=1),0,24000)
        ids=np.arange(24000,28831)
        append_results(family,"baseline_strict_D4","D4",ids,predict_artifact(a,ids))
        flush_results()
    show_family(family)
    show_family_diagnostics(family)


## 2. 求解、正则、对数与早期样条诊断（主7节）

7个拟合配置＋训练基率常数对照：baseline、C=0.1、C=10、log_counts、age_spline、
两者组合、recent_12000。baseline是原始全字段A的数值标准化版本，保留日历、宏观和原始pdays。
`log_counts`仅对campaign/previous取log1p再标准化。
早期求解使用`tol=1e-4,max_iter=5000`；该历史设置不被静默替换成严格设置。
每折从训练前缀得到常数 \(p_i=n_{\mathrm{yes}}/n\)，用于区分模型与基率基线。


In [9]:
display_table(family_parameters("diagnosis"))
run_standard("diagnosis")


model,representation,tol,max_iter,C,log_counts,spline,recent,constant
baseline,diagnosis,0.0001,5000.0,1.0,,,,
C_0.1,diagnosis,0.0001,5000.0,0.1,,,,
C_10,diagnosis,0.0001,5000.0,10.0,,,,
log_counts,diagnosis,0.0001,5000.0,1.0,True,,,
age_spline,diagnosis,0.0001,5000.0,1.0,,True,,
log_counts_age_spline,diagnosis,0.0001,5000.0,1.0,True,True,,
recent_12000,diagnosis,0.0001,5000.0,1.0,,,12000.0,
train_prior_dummy,,,,,,,,True


diagnosis : NOT_RUN


## 3. 严格求解复核（主7节）

在D4重拟合严格baseline，在最终val重拟合严格age_spline；
`C=1,tol=1e-7,max_iter=20000`，样条定义沿用早期5结点版本。
两项是不同窗口的求解复核，D4的baseline不会被注册成另一份最终test候选。


In [10]:
display_table(family_parameters("diagnosis_followup"))
run_standard("diagnosis_followup")


model,representation,C,spline
age_spline_strict,diagnosis,1,True


diagnosis_followup : NOT_RUN


## 4. 宏观字段消融（主7节）

M1保留5个宏观列，M2全部删除，M3只保留cons.price.idx、cons.conf.idx、euribor3m；
其余类别、日历与原始数值列保持一致。三者均`C=1,tol=1e-7,max_iter=20000`。
该实验改变输入集合，不是在估计宏观变量的因果效应。加上每折训练基率常数参照共4项。


In [11]:
display_table(family_parameters("macro"))
run_standard("macro")


model,representation,drop_macro,constant
M1_all_macro,macro,[],
M2_no_macro,macro,"[emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed]",
M3_three_macro,macro,"[emp.var.rate, nr.employed]",
train_prior_dummy,,,True


macro : NOT_RUN


## 5. 个人、历史与宏观联合表示（主7节）

P0_linear→P1_age_spline→P2_previous→P3_pdays_flag→P_full依次改变年龄及历史表示；
“P”系列仍含日历，不等于后来只保留个人字段的personal_only。
在P_full上加入标准化宏观列，对宏观块采用
\[
x_{\mathrm{macro}}'=x_{\mathrm{macro}}/\sqrt{\kappa},\qquad\kappa\in\{1,10,100\}.
\]
若保持原始尺度贡献不变，宏观系数的有效L2惩罚乘以 \(\kappa\)。
对P_full与M_penalty10分别删除age/job/education/marital/default/两贷款项/contact/month/
weekday/完整历史块/campaign，共22个配对删除配置。加8个递进/宏观配置及训练先验，共31项。
历史块共同出现且相关，删除实验不等于各列互不重叠的独立贡献。

运行P_full、M_penalty1/10/100时，还会自动输出参照类别和年龄对照：
\[
\Delta_{\ell,r}=\beta_\ell-\beta_r,\qquad
\mathrm{OR}_{\ell,r}=\exp(\Delta_{\ell,r}),\qquad
\Delta_{\mathrm{age},40}=[B(\mathrm{age})-B(40)]^\top\beta_{\mathrm{age}}.
\]
参照包括job=admin.、education=high.school、marital=married、poutcome=nonexistent等，
完整映射见辅助函数；缺少预定参照时按原实验采用训练编码中的首个类别。
年龄对照为20至80岁、每5岁一点，参照40岁。这些是条件log-odds对照，不是因果效应。
同时保留原来的11个字段在train/eval上的固定0.5分组指标，结果写入
`category_age_contrasts.csv` 和 `auxiliary_subgroups.csv`。


In [12]:
display_table(family_parameters("personal_macro"))
run_standard("personal_macro")


model,age,history,macro_penalty,drop,constant
P0_linear,linear,0.0,,[],
P1_age_spline,spline,0.0,,[],
P2_previous,spline,1.0,,[],
P3_pdays_flag,spline,2.0,,[],
P_full,spline,3.0,,[],
M_penalty1,spline,3.0,1.0,[],
M_penalty10,spline,3.0,10.0,[],
M_penalty100,spline,3.0,100.0,[],
P_drop_age,spline,3.0,,[age],
P_drop_job,spline,3.0,,[job],


personal_macro : NOT_RUN


## 6. 类别权重与年龄×职业交互（主7节）

两种表示P_full、M_penalty10 × 正类相对权重 \(r\in\{1,2,5,n_0/n_1\}\) × 有/无交互，共16项。
\[
\widetilde w_i=\begin{cases}r,&y_i=1\\1,&y_i=0,\end{cases}\qquad
w_i=\widetilde w_i/\overline{\widetilde w}.
\]
只在分类器fit时传入sample_weight；预处理不加权。固定交互列为
\[
\mathbf1[\mathrm{job=student}]\frac{\mathrm{age}-25}{10},\qquad
\mathbf1[\mathrm{job=retired}]\frac{\mathrm{age}-60}{10}.
\]
不按稀有职业频率再标准化；其余LR参数不变。保留固定0.5、本窗F1诊断及前窗F1迁移。
最终val还自动生成job、education在固定0.5和前窗F1阈值下的分组指标，以及编码列系数表。
如果只跑val而没有D4阈值，前窗规则明确记为不可用，不以本窗阈值替代。


In [13]:
display_table(family_parameters("weights"))
run_standard("weights")


model,age,history,macro_penalty,drop,yes_weight,interaction
P_w1_i0,spline,3,,[],1,False
P_w1_i1,spline,3,,[],1,True
P_w2_i0,spline,3,,[],2,False
P_w2_i1,spline,3,,[],2,True
P_w5_i0,spline,3,,[],5,False
P_w5_i1,spline,3,,[],5,True
P_wbalanced_i0,spline,3,,[],balanced,False
P_wbalanced_i1,spline,3,,[],balanced,True
M10_w1_i0,spline,3,10.0,[],1,False
M10_w1_i1,spline,3,10.0,[],1,True


weights : NOT_RUN


## 7. 早期时间校准（主7节）

每个训练前缀长度为n，基模型只用前 \(\lfloor0.8n\rfloor\) 条fit，
后20%作为校准段，随后原D1–D4/val作为评价段。不在校准后重新拟合基模型。
P/M10/M100使用相同后期history表示与宏观惩罚，构成3×3=9个输出版本。

原始LR线性分数记为z：
\[
p_{\rm raw}=\sigma(z),\quad
p_{\rm intercept}=\sigma(z+b),\quad
p_{\rm sigmoid}=\sigma(az+b).
\]
截距b在[-10,10]内最小化校准段平均log loss，`bounded,xatol=1e-10`；
sigmoid是在z上拟合一元LR，`C=1,tol=1e-9,max_iter=20000`；
若斜率a≤0则回退恒等映射。保留raw对照。
阈值在校准段按变换后的概率重新选择F1最优；不使用评价段标签选阈值。

下面的映射实现也用于后面的val前向校准；后者的加权目标保留原始sum形式，
早期方案保留mean形式，以避免改变有界优化的数值细节。


In [14]:
def apply_mapping(p,slope=1.,offset=0.,z=None):
    if slope==1 and offset==0 and z is None:return np.asarray(p).copy()
    if z is None:
        eps=np.finfo(float).eps
        z=logit(np.clip(p,eps,1-eps))
    return expit(slope*np.asarray(z)+offset)

def fit_mapping(p,y,weight,method,z=None,mean_objective=False):
    assert not TEST_STARTED
    y,weight=np.asarray(y),np.asarray(weight)
    if z is None:
        eps=np.finfo(float).eps;z=logit(np.clip(p,eps,1-eps))
    fallback=False;raw_slope=None
    if method=="raw":slope,offset=1.,0.
    elif method=="intercept_only":
        def objective(b):
            loss=np.logaddexp(0,z+b)-y*(z+b)
            return float(np.mean(loss)) if mean_objective else float(np.dot(weight,loss))
        opt=minimize_scalar(objective,bounds=(-10,10),method="bounded",options={"xatol":1e-10})
        assert opt.success
        slope,offset=1.,float(opt.x)
    else:
        assert method=="sigmoid_positive_slope"
        lr=logistic(C=1,tol=1e-9,max_iter=20000)
        with threadpool_limits(limits=1):
            if mean_objective:
                lr.fit(np.asarray(z).reshape(-1,1),y)
            else:
                lr.fit(np.asarray(z).reshape(-1,1),y,sample_weight=weight)
        raw_slope=float(lr.coef_[0,0]);fallback=raw_slope<=0
        slope,offset=(1.,0.) if fallback else (raw_slope,float(lr.intercept_[0]))
    return dict(slope=slope,offset=offset,identity_fallback=fallback,raw_slope=raw_slope,
                fit_n=len(y),weighted_prior=float(np.dot(weight,y)/weight.sum()),
                natural_prior=float(y.mean()),ess=float(weight.sum()**2/np.dot(weight,weight)))

PARAMETER_TABLES={
    "early_calibration":[dict(base=b,macro_penalty=k,fit_fraction=.8,calibration="remaining training prefix",
        method=m,base_C=1,base_tol=1e-7,base_max_iter=20000,
        sigmoid_C=1 if m=="sigmoid_positive_slope" else "",
        sigmoid_tol=1e-9 if m=="sigmoid_positive_slope" else "",
        offset_bounds="[-10,10]" if m=="intercept_only" else "")
        for b,k in [("P",None),("M10",10),("M100",100)]
        for m in ["raw","intercept_only","sigmoid_positive_slope"]],
    "calibration":[dict(base=b,policy=w,method=m,sliding_n=2060 if w=="S" else "",
        half_life_rows=1030 if w=="W" else "",
        sigmoid_C=1 if m=="sigmoid_positive_slope" else "",
        sigmoid_tol=1e-9 if m=="sigmoid_positive_slope" else "",
        offset_bounds="[-10,10]" if m=="intercept_only" else "")
        for b in ["LR_N","LR_W","LR_O2","RF"] for w in ["E","S","W"]
        for m in ["raw","intercept_only","sigmoid_positive_slope","prior_natural","prior_weighted"]],
    "kalman":[dict(method=m,base_C=.01,alpha=.5 if m=="ewma" else "",
        Q=.1 if m=="kalman" else "",initial_variance=1. if m=="kalman" else "",
        warmup="V1-V2",evaluation="V3-V6")
              for m in ["raw","fixed_intercept","ewma","kalman"]],
}

def run_early_calibration():
    family="early_calibration"
    if family not in EXPERIMENTS:print(family,": NOT_RUN");return
    for fold,stop,end in FOLDS:
        if fold not in FOLD_NAMES:continue
        cut=int(stop*.8)
        for arm,k in [("P",None),("M10",10),("M100",100)]:
            artifact=fit_bundle(dict(age="spline",history=3,macro_penalty=k,drop=[]),0,cut)
            cal_ids=np.arange(cut,stop); ids=np.arange(stop,end)
            zc=predict_artifact(artifact,cal_ids,True);zv=predict_artifact(artifact,ids,True)
            for method in ["raw","intercept_only","sigmoid_positive_slope"]:
                name=arm+"__"+method
                if family in MODEL_FILTER and name not in MODEL_FILTER[family]:continue
                par=fit_mapping(expit(zc),Y_DEV[cal_ids],np.ones(len(cal_ids)),method,z=zc,mean_objective=True)
                cp=apply_mapping(expit(zc),par["slope"],par["offset"],zc)
                q=apply_mapping(expit(zv),par["slope"],par["offset"],zv)
                t=choose_f1_threshold(Y_DEV[cal_ids],cp)
                PARAMETERS.append(dict(family=family,model=name,fold=fold,fit_start=cut,fit_stop=stop,**par,threshold=t))
                append_results(family,name,fold,ids,q,{"calibration_F1":t})
                if fold=="val":
                    register_final(family,name,artifact,{"saved_calibration_F1":t},slope=par["slope"],
                                   offset=par["offset"],use_decision=True)
        flush_results()
    show_family(family)

def run_forward_calibration():
    family="calibration"
    if family not in EXPERIMENTS:print(family,": NOT_RUN");return
    for base in ["LR_N","LR_W","LR_O2","RF"]:
        artifact=fit_bundle(DIAGNOSTIC_BASES[base],0,28831)
        all_ids=np.arange(28831,35009);pall=predict_artifact(artifact,all_ids)
        for fi in range(4):
            stop,end=int(VAL_BOUNDS[fi+2]),int(VAL_BOUNDS[fi+3])
            for policy in ["E","S","W"]:
                start=stop-2060 if policy=="S" else 28831
                fit_ids=np.arange(start,stop);ids=np.arange(stop,end)
                fp=pall[fit_ids-28831];p=pall[ids-28831];fy=Y_DEV[fit_ids]
                w=np.exp2(-(fit_ids.max()-fit_ids)/1030.) if policy=="W" else np.ones(len(fit_ids))
                w=w*len(w)/w.sum()
                for method in ["raw","intercept_only","sigmoid_positive_slope","prior_natural","prior_weighted"]:
                    name=f"{base} | {policy} | {method}"
                    if family in MODEL_FILTER and name not in MODEL_FILTER[family]:continue
                    if method.startswith("prior_"):
                        value=float(fy.mean()) if method=="prior_natural" else float(np.dot(w,fy)/w.sum())
                        q=np.full(len(ids),value);extra={}
                        par=dict(constant=value,fit_n=len(fy))
                    else:
                        par=fit_mapping(fp,fy,w,method)
                        fq=apply_mapping(fp,par["slope"],par["offset"])
                        q=apply_mapping(p,par["slope"],par["offset"])
                        t=choose_f1_threshold(fy,fq);extra={"fit_F1":t}
                        assert np.array_equal(np.lexsort((ids,-p)),np.lexsort((ids,-q)))
                    PARAMETERS.append(dict(family=family,model=name,fold=f"F{fi+1}",
                                           fit_start=start,fit_stop=stop,**par,**extra))
                    append_results(family,name,f"F{fi+1}",ids,q,extra)
                    if fi==3:
                        mapping={"constant":par["constant"]} if "constant" in par else {
                            "slope":par["slope"],"offset":par["offset"]}
                        register_final(family,name,artifact,{"saved_"+k:v for k,v in extra.items()},**mapping)
        flush_results()
    show_family(family,"F4")


In [15]:
display_table(family_parameters("early_calibration"))
run_early_calibration()


base,macro_penalty,fit_fraction,calibration,method,base_C,base_tol,base_max_iter,sigmoid_C,sigmoid_tol,offset_bounds
P,NaN,0.8,remaining training prefix,raw,1,1.000000e-07,20000,,,
P,NaN,0.8,remaining training prefix,intercept_only,1,1.000000e-07,20000,,,"[-10,10]"
P,NaN,0.8,remaining training prefix,sigmoid_positive_slope,1,1.000000e-07,20000,1,0.0,
M10,10.0,0.8,remaining training prefix,raw,1,1.000000e-07,20000,,,
M10,10.0,0.8,remaining training prefix,intercept_only,1,1.000000e-07,20000,,,"[-10,10]"
M10,10.0,0.8,remaining training prefix,sigmoid_positive_slope,1,1.000000e-07,20000,1,0.0,
M100,100.0,0.8,remaining training prefix,raw,1,1.000000e-07,20000,,,
M100,100.0,0.8,remaining training prefix,intercept_only,1,1.000000e-07,20000,,,"[-10,10]"
M100,100.0,0.8,remaining training prefix,sigmoid_positive_slope,1,1.000000e-07,20000,1,0.0,


early_calibration : NOT_RUN


## 8. 信息范围对照（主7–8节）

personal_only只用age、job、marital、education、default、housing、loan。
personal_contact_history在其上加入contact、campaign、previous、pdays、poutcome的历史表示。
二者均排除month、day_of_week、全部宏观列与duration，年龄为后期4结点样条。
这里C=1；主文最终联系LR的C=0.01来自下一节的配置网格，不能混称同一个模型。


In [16]:
display_table(family_parameters("scope"))
run_standard("scope")


model,age,history,macro_penalty,drop,personal_only,C
personal_only,spline,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,
personal_contact_history,spline,3,,"[month, day_of_week]",,1.0


scope : NOT_RUN


## 9. LR与浅树的Precision约束（主7–8节）

已有两个信息范围对照，加16个LR与12棵浅树，共30项。
LR：个人/联系 × linear/spline年龄 × \(C\in\{0.01,0.1,1,10\}\)。
树：个人/联系 × depth∈{2,3,4} × min_samples_leaf∈{100,300}，
`criterion="gini",random_state=42`，年龄线性标准化。
目标Precision为0.3/0.4/0.5/0.6，最少支持分别检查1和30；30是经验支持数，不是置信保证。
选择规则见1.1。前窗阈值跨模型重拟合后的迁移可能因概率尺度变化而失效，仍按实际结果报告。


In [17]:
display_table(family_parameters("precision"))
run_standard("precision")


model,age,history,macro_penalty,drop,personal_only,C,kind,depth,leaf
personal_only,spline,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,,,,
personal_contact_history,spline,3,,"[month, day_of_week]",,1.0,,,
personal_LR_linear_C0.01,linear,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,0.01,,,
personal_LR_linear_C0.1,linear,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,0.1,,,
personal_LR_linear_C1,linear,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,1.0,,,
personal_LR_linear_C10,linear,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,10.0,,,
personal_LR_spline_C0.01,spline,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,0.01,,,
personal_LR_spline_C0.1,spline,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,0.1,,,
personal_LR_spline_C1,spline,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,1.0,,,
personal_LR_spline_C10,spline,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,10.0,,,


precision : NOT_RUN


## 10. 原HGB扩展支线（主附录A）

个人/联系 × depth∈{2,3} × L2∈{1,10}，共8项。
`learning_rate=0.05,max_iter=150,min_samples_leaf=100,max_leaf_nodes=2**depth`，
`early_stopping=False,random_state=42,class_weight=None,categorical_features=None`。
使用相同历史表示，年龄线性标准化，编码转稠密，不包含日历、宏观或duration。

加法更新为 \(F_m(x)=F_{m-1}(x)+\eta h_m(x)\)，输出 \(\sigma(F_m(x))\)；
树叶正则由l2_regularization控制。保留相同Precision约束。
原错误分析的Top100/200/500包含第k名的全部同分者；它与本文件精确预算Top5/10/20%不同，
辅助函数 `top_k_tied_diagnostic` 明确保留该规则。
最终val的HGB运行器会自动输出Top100/200/500及其TP/FP人群构成；
第9节运行personal_LR_spline_C0.01、contact_LR_linear_C0.01、
contact_LR_spline_C0.01时也会输出原先用于对照的同类诊断。
结果保存在 `top_groups.csv` 与 `high_score_composition.csv`，只描述开发val。


In [18]:
display_table(family_parameters("boosting"))
run_standard("boosting")


model,age,history,macro_penalty,drop,personal_only,dense,kind,depth,leaf,l2,learning_rate,iterations,loss,max_bins,max_features
personal_HGB_d2_l21,linear,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,True,HGB,2,100,1,0.05,150,log_loss,255,1.0
personal_HGB_d2_l210,linear,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,True,HGB,2,100,10,0.05,150,log_loss,255,1.0
personal_HGB_d3_l21,linear,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,True,HGB,3,100,1,0.05,150,log_loss,255,1.0
personal_HGB_d3_l210,linear,0,,"[contact, month, day_of_week, poutcome, previous, pdays, campaign]",True,True,HGB,3,100,10,0.05,150,log_loss,255,1.0
contact_HGB_d2_l21,linear,3,,"[month, day_of_week]",,True,HGB,2,100,1,0.05,150,log_loss,255,1.0
contact_HGB_d2_l210,linear,3,,"[month, day_of_week]",,True,HGB,2,100,10,0.05,150,log_loss,255,1.0
contact_HGB_d3_l21,linear,3,,"[month, day_of_week]",,True,HGB,3,100,1,0.05,150,log_loss,255,1.0
contact_HGB_d3_l210,linear,3,,"[month, day_of_week]",,True,HGB,3,100,10,0.05,150,log_loss,255,1.0


boosting : NOT_RUN


## 11. campaign/pdays分箱（主7节）

P_full、M_penalty10 × current/campaign_only/pdays_only/both，共8项，只做原train→val。
campaign变为1、2、3、4plus；pdays变为999、0、1–7、8–30、>30。
使用campaign箱时删除campaign_log；使用pdays箱时同时删除pdays_log和pdays_known，
避免把旧表示与新表示叠加。previous_group及poutcome不变。
分箱会改变维度和有效正则结构，不能把差异全部归因于“离散化本身”。
保留固定0.5、联系预算和本val精确并列F1诊断。


In [19]:
display_table(family_parameters("binning"))
run_standard("binning")


model,age,history,macro_penalty,drop,campaign_bin,pdays_bin
P_full__current,spline,3,,[],False,False
P_full__campaign_only,spline,3,,[],True,False
P_full__pdays_only,spline,3,,[],False,True
P_full__both,spline,3,,[],True,True
M_penalty10__current,spline,3,10.0,[],False,False
M_penalty10__campaign_only,spline,3,10.0,[],True,False
M_penalty10__pdays_only,spline,3,10.0,[],False,True
M_penalty10__both,spline,3,10.0,[],True,True


binning : NOT_RUN


## 12. 五个LR与三类RF的不平衡实验（主9.1节）

五个LR基模型：contact_LR_spline_C0.01、M_drop_weekday、P_drop_weekday、
M2_no_macro、diagnosis_age_spline。每个模型：N无干预1项、W类平衡1项，
O2/O1/U2/U1各3种子（17/42/2026），共14项；5×14=70项。
普通RF、加权RF、BRF各3种子，再加9项，总计79项。

| 策略 | 训练操作 | 训练权重 |
|---|---|---|
| N | 保持样本 | 全1 |
| W | 保持样本 | \(w_c=n/(2n_c)\) |
| O2 / O1 | 保留全部原样本，正类有放回补到 \(\lceil n_0/2\rceil\) / \(n_0\) | 每个采样后样本 \(n/m\) |
| U2 / U1 | 保留全部正类，负类无放回抽到 \(2n_1\) / \(n_1\) | 每个采样后样本 \(n/m\) |

这里n是原训练行数、m是采样后行数，所以 \(\sum w_i=n\)，避免单纯改变总权重带来的正则变化。
采样用 `np.random.default_rng(seed)`；欠采样后按原索引排序，过采样保持原样本后追加抽样索引。
BRF每棵树内部均衡采样，`replacement=True,bootstrap=False`；
加权RF的balanced_subsample只改变每棵bootstrap树的类别权重。
三类RF的其余参数见1.3。保存开发集Recall≥0.2/0.4/0.6所需的最高同分边界阈值。

三个阈值按D1→D2→D3→D4→val顺序迁移：本窗选阈值的结果标为
`same_window_SELECTION_ONLY_recall*`；下一窗使用前窗选出的阈值，标为
`previous_window_recall_TRANSFER_target*`。`thresholds.csv`记录来源折和是否可用。
最终val模型冻结时，同时保留本val选择规则和原D4阈值；随后若显式开启test，
`saved_dev_previous_window_recall_TRANSFER_target*`消费的是D4阈值，不会在test上重选。
只运行val时没有D4迁移阈值，这三项记为`NO_PREVIOUS_WINDOW`并不评分；要复现迁移结果，
至少依序运行 `FOLD_NAMES=["D4","val"]`。


In [20]:
display_table(family_parameters("imbalance"))
run_standard("imbalance")


model,age,history,macro_penalty,drop,C,sampling,seed,representation,drop_macro,spline,kind,weighted_rf
contact_LR_spline_C0.01 | none | seed=0,spline,3.0,,"[month, day_of_week]",0.01,none,0,,,,,
contact_LR_spline_C0.01 | balanced | seed=0,spline,3.0,,"[month, day_of_week]",0.01,balanced,0,,,,,
contact_LR_spline_C0.01 | ros_1to2 | seed=17,spline,3.0,,"[month, day_of_week]",0.01,ros_1to2,17,,,,,
contact_LR_spline_C0.01 | ros_1to2 | seed=42,spline,3.0,,"[month, day_of_week]",0.01,ros_1to2,42,,,,,
contact_LR_spline_C0.01 | ros_1to2 | seed=2026,spline,3.0,,"[month, day_of_week]",0.01,ros_1to2,2026,,,,,
contact_LR_spline_C0.01 | ros_1to1 | seed=17,spline,3.0,,"[month, day_of_week]",0.01,ros_1to1,17,,,,,
contact_LR_spline_C0.01 | ros_1to1 | seed=42,spline,3.0,,"[month, day_of_week]",0.01,ros_1to1,42,,,,,
contact_LR_spline_C0.01 | ros_1to1 | seed=2026,spline,3.0,,"[month, day_of_week]",0.01,ros_1to1,2026,,,,,
contact_LR_spline_C0.01 | rus_1to2 | seed=17,spline,3.0,,"[month, day_of_week]",0.01,rus_1to2,17,,,,,
contact_LR_spline_C0.01 | rus_1to2 | seed=42,spline,3.0,,"[month, day_of_week]",0.01,rus_1to2,42,,,,,


imbalance : NOT_RUN


## 13. poutcome列尺度（主9.2节）

固定联系LR `C=0.01`，仅将编码后的poutcome列乘 \(s\in\{0.5,1,2\}\)。
同一原尺度线性贡献下，
\[
x'=sx,\quad\beta'=\beta/s,\quad
\|\beta'\|^2=\|\beta\|^2/s^2.
\]
这是有效L2惩罚的改变，不是强行给历史成功更高预测概率；符号和系数仍由训练决定。
本文件从原训练样本重建相同预处理；历史实验复用了该预处理，数值一致性在开发验证中检查。


In [21]:
display_table(family_parameters("history_scale"))
run_standard("history_scale")


model,age,history,macro_penalty,drop,C,sampling,seed,poutcome_scale
poutcome_scale0.5,spline,3,,"[month, day_of_week]",0.01,none,0,0.5
poutcome_scale1.0,spline,3,,"[month, day_of_week]",0.01,none,0,1.0
poutcome_scale2.0,spline,3,,"[month, day_of_week]",0.01,none,0,2.0


history_scale : NOT_RUN


### 描述性历史统计

以下调用使用1.5节已定义的辅助函数，只访问开发数据；不会调整模型参数。


## 14. 历史信息诊断基模型（主9.2节）

8个已定义模型：LR_N、LR_W、LR_O2(seed42)、RF(seed42)、M_drop_weekday、
P_drop_weekday、M2_no_macro和diagnosis_age_spline。它们复用相同配置与训练范围，不是8种新算法。
只评价原val；概率与全体名单的poutcome分组指标由公共执行器给出。
`subgroup_history_diagnostics()`提供train/val中poutcome、previous、pdays的支持数和正类率，
用于检查小样本和时期差异，不能把这些描述当作因果效应。


In [22]:
display_table(family_parameters("history_diagnostic"))
run_standard("history_diagnostic")


model,age,history,macro_penalty,drop,C,sampling,seed,kind,weighted_rf,representation,drop_macro,spline
LR_N,spline,3.0,,"[month, day_of_week]",0.01,none,0,,,,,
LR_W,spline,3.0,,"[month, day_of_week]",0.01,balanced,0,,,,,
LR_O2,spline,3.0,,"[month, day_of_week]",0.01,ros_1to2,42,,,,,
RF,spline,3.0,,"[month, day_of_week]",,,42,RF,False,,,
M_drop_weekday,spline,3.0,10.0,[day_of_week],,none,0,,,,,
P_drop_weekday,spline,3.0,,[day_of_week],,none,0,,,,,
M2_no_macro,,,,,,none,0,,,macro,"[emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed]",
diagnosis_age_spline,,,,,1.0,none,0,,,diagnosis,,True


history_diagnostic : NOT_RUN


In [23]:
if "history_diagnostic" in EXPERIMENTS:
    display_table(subgroup_history_diagnostics())


## 15. val上的E/S/W前向校准（主9.2节）

基模型固定在原train训练：LR_N、LR_W、LR_O2(seed42)、普通RF(seed42)。
val按原始行号分成互不重叠的V1–V6：
`[28831,29861),[29861,30891),[30891,31921),[31921,32951),[32951,33980),[33980,35009)`。
但**校准拟合窗口可重叠**：每次只用当前评价块之前的标签，评价V3–V6，记为F1–F4。

| 策略 | 拟合范围 | 样本权重 |
|---|---|---|
| E | [28831, 当前块起点) | 全1 |
| S | 当前块之前恰好2060行 | 全1 |
| W | 与E相同 | 半衰期1030行，归一化均值1 |

\[
\widetilde w_i=2^{-(r_{\max}-r_i)/1030},\qquad
w_i=\widetilde w_i/\overline{\widetilde w},\qquad
\mathrm{ESS}=\frac{(\sum_iw_i)^2}{\sum_iw_i^2}.
\]
S以行数2060为准，后面分块长度为1029时其边界并非恰好两个完整V块。
每个base×policy输出raw、intercept_only、sigmoid_positive_slope、prior_natural、
prior_weighted，共4×3×5=60个配置/参照、4个评价窗。
先验常数分别为 \(\bar y\) 与 \(\sum w_i y_i/\sum w_i\)，不是保序映射。

输入z为裁剪到 \([\epsilon,1-\epsilon]\) 后概率的logit。
截距优化目标是加权log loss之和；其余参数与第7节相同。当前实现**没有isotonic**。
在实数运算下，严格正斜率映射保留同一评价块的严格次序与原有同分关系。
实际浮点计算中的概率裁剪、舍入或sigmoid饱和可能把不同分数变成完全同分，
随后名单会受行号破同分规则影响；此时AP/AUC/Top-k不保证完全不变。
已有同分只有在前后同分分组和破同分规则一致时才可比较。常数先验不属于严格保序映射。
代码保留实际排序一致断言，以便遇到例外时明确报错。F1阈值需要转换/重选。
跨块混合不同映射后的概率也不保证排序不变。

最终test使用**F4时已经拟合好的映射**，其拟合数据截止33980之前；
V6仍是原开发评价块。本文件不偷偷用完整val再拟合另一套校准器。


In [24]:
display_table(family_parameters("calibration"))
run_forward_calibration()


base,policy,method,sliding_n,half_life_rows,sigmoid_C,sigmoid_tol,offset_bounds
LR_N,E,raw,,,,,
LR_N,E,intercept_only,,,,,"[-10,10]"
LR_N,E,sigmoid_positive_slope,,,1,0.0,
LR_N,E,prior_natural,,,,,
LR_N,E,prior_weighted,,,,,
LR_N,S,raw,2060,,,,
LR_N,S,intercept_only,2060,,,,"[-10,10]"
LR_N,S,sigmoid_positive_slope,2060,,1,0.0,
LR_N,S,prior_natural,2060,,,,
LR_N,S,prior_weighted,2060,,,,


calibration : NOT_RUN


## 16. 最后四个单因素LR方案（主10节）

固定联系LR `C=0.01,tol=1e-7,max_iter=20000`，分别比较baseline、
pdays_coarse、drop_loans、time_weighted；不把改动叠加。
pdays_coarse使用第11节的pdays箱并移除旧pdays表示；drop_loans删除housing/loan。
时间加权在每个训练窗内部按行龄设置：
\[
a_i=n-1-i,\qquad\widetilde w_i=2^{-a_i/n},\qquad
w_i=\widetilde w_i/\overline{\widetilde w}.
\]
最新与最早样本的相对权重接近2，预处理仍不加权。五窗训练后比较预算指标和概率误差，
F1阈值只在开发窗确定，不能根据test改半衰期。


In [25]:
display_table(family_parameters("refinement"))
run_standard("refinement")


model,age,history,macro_penalty,drop,C,pdays_bin,time_weighted
baseline,spline,3,,"[month, day_of_week]",0.01,,
pdays_coarse,spline,3,,"[month, day_of_week]",0.01,True,
drop_loans,spline,3,,"[month, day_of_week, housing, loan]",0.01,,
time_weighted,spline,3,,"[month, day_of_week]",0.01,,True


refinement : NOT_RUN


## 17. 全历史与最近12,000条的RF/HGB（主10.1节）

RF/HGB × full_history/recent_12000 × D1–D4/val。
每一训练窗口内部重新fit编码器、标准化和后期4结点年龄样条。
D1的两种训练范围都是[0,12000)，属于恒等检查，不是独立重复拟合。

| 参数 | RF | 本节HGB |
|---|---|---|
| 树数/迭代数 | 300 | 100 |
| max_depth | None | 2 |
| min_samples_leaf | 5 | 20 |
| max_features / max_leaf_nodes | sqrt | 31 |
| learning_rate / L2 | 不适用 | 0.1 / 1 |
| loss / max_bins / max_features | 不适用 / 不适用 / sqrt | log_loss / 255 / 1.0 |
| bootstrap / early_stopping | True | False |
| random_state | 42 | 42 |

**本节HGB不同于第10节的原HGB网格**，不可混用150轮、0.05学习率、叶节点支持100等参数。
缩短窗口同时减少训练样本量并提高近期样本占比，因此配对差异不是纯粹的时间权重效应。


In [26]:
display_table(family_parameters("window"))
run_standard("window")


model,age,history,macro_penalty,drop,kind,dense,seed,weighted_rf,recent,depth,max_leaf_nodes,leaf,l2,learning_rate,iterations,loss,max_bins,max_features
RF | full_history,spline,3,,"[month, day_of_week]",RF,True,42.0,False,,,,,,,,,,
RF | recent_12000,spline,3,,"[month, day_of_week]",RF,True,42.0,False,12000.0,,,,,,,,,
HGB | full_history,spline,3,,"[month, day_of_week]",HGB,True,,,,2.0,31.0,20.0,1.0,0.1,100.0,log_loss,255.0,1.0
HGB | recent_12000,spline,3,,"[month, day_of_week]",HGB,True,,,12000.0,2.0,31.0,20.0,1.0,0.1,100.0,log_loss,255.0,1.0


window : NOT_RUN


## 18. 动态截距与卡尔曼式校准（主9.3节）

固定原train联系LR，`C=0.01`；val V1–V2用于预热，V3–V6按“先发预测、再接收该批标签”评价。
该模型对伯努利计数观测作局部线性化，属于卡尔曼式近似，不是直接对二元标签应用精确线性高斯滤波。

设批次t开始前的截距状态为 \(m_{t-1}\)，方差 \(P_{t-1}\)，基础分数为 \(z_i\)：
\[
P_t^-=P_{t-1}+Q,\quad Q=0.1,\qquad
p_{it}^-=\sigma(z_i+m_{t-1}).
\]
先保存全批预测；收到该批标签后定义
\[
h_t=\sum_i p_{it}^-,\quad
H_t=\sum_i p_{it}^-(1-p_{it}^-),\quad R_t=H_t,\quad
e_t=\sum_i y_i-h_t.
\]
\[
K_t=\frac{P_t^-H_t}{H_t^2P_t^-+R_t},\qquad
m_t=m_{t-1}+K_te_t,
\]
\[
P_t=(1-K_tH_t)^2P_t^-+K_t^2R_t.
\]
初始 \(m_0=0,P_0=1\)。Q固定，没有按test调参。

对照包括raw、V1–V2拟合后保持不变的固定截距，以及EWMA：
\[
\widehat b_t=\arg\min_{b\in[-10,10]}\sum_i
[\log(1+\exp(z_i+b))-y_i(z_i+b)],\quad
b_t^{\rm EWMA}=0.5b_{t-1}^{\rm EWMA}+0.5\widehat b_t.
\]
`intercept_mle`用一阶条件 \(\sum\sigma(z_i+b)=\sum y_i\) 的有界求根；
没有内部根时采用相应边界。标签在下一批前全部返回是模拟假设。

**test边界：** raw用0；固定截距用V1–V2结果；EWMA/Kalman用V6反馈后的终态。
整段test保持一个固定截距，不读取test标签更新状态。因此test上的排序、AUC、AP和Top-k名单相同，
概率误差与固定0.5操作点可以不同；它评价的是终态迁移，不是带test反馈的在线滤波。
上述排序结论同样以未因裁剪或浮点饱和产生新同分为前提，具体例外见第15节。


In [27]:
def intercept_mle(z,y):
    f=lambda b:float(expit(z+b).sum()-np.sum(y))
    if f(-10)>=0:return -10.
    if f(10)<=0:return 10.
    return float(brentq(f,-10.,10.,xtol=1e-13))

def sequential_offsets(p_raw,labels,callback=None):
    z_all=logit(np.asarray(p_raw))
    relative=VAL_BOUNDS-28831
    m,variance,ema,fixed=0.,1.,0.,None
    forecasts,states=[],[]
    for block in range(6):
        lo,hi=map(int,relative[block:block+2]);z=z_all[lo:hi]
        prior_variance=variance+.1
        offsets={"raw":0.,"ewma":ema,"kalman":m}
        if block>=2:offsets["fixed_intercept"]=fixed
        predictions={method:(np.asarray(p_raw)[lo:hi].copy() if method=="raw" else expit(z+b))
                     for method,b in offsets.items()}
        # Forecasts are constructed before this block's labels are read.
        for method,b in offsets.items():
            forecasts.append(dict(method=method,block=f"V{block+1}",lo=lo,hi=hi,
                                  p=predictions[method],offset=b,scored=block>=2))
        if callback:callback(block,predictions)
        yy=np.asarray(labels[lo:hi],dtype=np.int8)
        target=intercept_mle(z,yy)
        old_ema,old_m=ema,m
        ema=.5*ema+.5*target
        mu=predictions["kalman"];h=float(mu.sum());H=float(np.sum(mu*(1-mu)));R=H
        gain=prior_variance*H/(H*H*prior_variance+R)
        innovation=float(yy.sum()-h)
        m=m+gain*innovation
        variance=(1-gain*H)**2*prior_variance+gain*gain*R
        assert np.isfinite([m,variance,ema]).all() and variance>0
        states.append(dict(block=f"V{block+1}",block_mle=target,ema_prior=old_ema,ema_posterior=ema,
                           kalman_prior=old_m,kalman_posterior=m,variance_prior=prior_variance,
                           variance_posterior=variance,predicted_yes=h,H=H,R=R,gain=gain,innovation=innovation))
        if block==1:fixed=intercept_mle(z_all[:relative[2]],np.asarray(labels[:relative[2]]))
    return forecasts,pd.DataFrame(states),dict(raw=0.,fixed_intercept=fixed,ewma=ema,kalman=m)

def run_kalman():
    family="kalman"
    if family not in EXPERIMENTS:print(family,": NOT_RUN");return
    artifact=fit_bundle(CONTACT,0,28831)
    ids=np.arange(28831,35009);p=predict_artifact(artifact,ids)
    forecasts,states,offsets=sequential_offsets(p,Y_DEV[ids])
    for forecast in forecasts:
        if not forecast["scored"]:continue
        if family in MODEL_FILTER and forecast["method"] not in MODEL_FILTER[family]:continue
        bid=ids[forecast["lo"]:forecast["hi"]]
        append_results(family,forecast["method"],forecast["block"],bid,forecast["p"])
        PARAMETERS.append(dict(family=family,model=forecast["method"],fold=forecast["block"],
                               offset=forecast["offset"],prediction_precedes_feedback=True))
    states.to_csv(OUTPUT_DIR/"kalman_states.csv",index=False)
    for method,b in offsets.items():
        if family in MODEL_FILTER and method not in MODEL_FILTER[family]:continue
        register_final(family,method,artifact,slope=1.,offset=b)
    flush_results()
    show_family(family,"V6")


In [28]:
display_table(family_parameters("kalman"))
run_kalman()


method,base_C,alpha,Q,initial_variance,warmup,evaluation
raw,0.01,,,,V1-V2,V3-V6
fixed_intercept,0.01,,,,V1-V2,V3-V6
ewma,0.01,0.5,,,V1-V2,V3-V6
kalman,0.01,,0.1,1.0,V1-V2,V3-V6


kalman : NOT_RUN


## 19. 固定方案的test评价与结果使用

下面只消费上述开发阶段已经生成的模型、校准参数和阈值。
没有新的fit、网格搜索、阈值选择或状态更新。默认开关关闭。
同一个基模型的test概率复用；不同校准方式由冻结映射计算。
本文件不自动覆盖主 Notebook 的历史表格；请把新生成结果和原结果区分开。

输出 `METRICS` 包含所有统一指标；`GROUP_METRICS`包含poutcome切片；
`PARAMETERS`包含实际拟合的校准参数/状态，`THRESHOLD_LOG`保留不可达约束。
`FINAL_MODELS`显示哪些最终模型进入了本次test评价，不包括D1–D4中间模型。
`CONTRASTS`、`AUX_GROUPS`、`COEFFICIENTS`、`TOPK_DIAGNOSTICS`和`TOPK_COMPOSITION`
分别保留原实验的参照对照、附加人群切片、系数、含边界同分的Top-k和TP/FP构成。
这些附加诊断按原协议在开发阶段生成，不会为了补表重新拟合或读取test。
本文件不构造“从test中选出的最佳模型”，也不把事后补测写成独立确认。


In [29]:
def evaluate_frozen_test():
    global TEST_STARTED
    if not RUN_TEST:
        print("Test evaluation: NOT_RUN. RUN_TEST=False.")
        return
    assert FINAL_MODELS, "First enable development families and run their cells."
    assert not TEST_STARTED, "Test evaluation has already run in this kernel."
    frozen={key:dict(value) for key,value in FINAL_MODELS.items()}
    TEST_STARTED=True
    # All model fits, calibration fits, thresholds and terminal states exist before this read.
    with DATA_PATH.open("rb",buffering=0) as handle:
        header=handle.readline();handle.seek(DEV_BYTE_STOP);payload=handle.read()
    data=pd.read_csv(io.BytesIO(header+payload),sep=";")
    assert data.shape==(6179,21)
    ids=np.arange(35009,41188);data.index=ids
    X=data.drop(columns=["y","duration"])
    probability_cache,decision_cache={},{}
    for (family,name),frozen_model in frozen.items():
        artifact=frozen_model["artifact"]
        if "constant" in frozen_model:
            p=np.full(len(data),frozen_model["constant"])
        else:
            if artifact not in probability_cache:
                bundle=joblib.load(ARTIFACTS[artifact]["path"])
                with threadpool_limits(limits=1):
                    probability_cache[artifact]=prediction_from_bundle(bundle,X)
                    if any(v["artifact"]==artifact and v.get("use_decision") for v in frozen.values()):
                        decision_cache[artifact]=prediction_from_bundle(bundle,X,True)
                del bundle;gc.collect()
            p=probability_cache[artifact]
            if "slope" in frozen_model:
                p=apply_mapping(p,frozen_model["slope"],frozen_model["offset"],
                                decision_cache[artifact] if frozen_model.get("use_decision") else None)
        append_results(family,name,"test",ids,p,frozen_model["rules"],data)
    assert frozen==FINAL_MODELS
    flush_results()
    result=pd.DataFrame(METRICS)
    result=result[result.split.eq("test")]
    display_table(result[result.rule.eq("top10")])
    print("POSTHOC_SUPPLEMENTARY_NOT_INDEPENDENT_CONFIRMATION",
          "| Test fitting: 0 | Test state updates: 0 | Metric rows:",len(result))


In [30]:
evaluate_frozen_test()
if METRICS:
    results = pd.DataFrame(METRICS)
    display(results.head(10))
    print("Calculated metric rows:", len(results))
else:
    print("No experiments were selected. Method definitions and all parameter tables are available above.")


Test evaluation: NOT_RUN. RUN_TEST=False.
No experiments were selected. Method definitions and all parameter tables are available above.


## 20. 复现检查与使用限制

- 数据切分、样本数、标签编码、权重归一化和参数网格在代码内明确固定。
- 上述所有执行函数都在本文件定义；不需要导入本地实验模块。
- 默认Run All只检查加载与定义；打开支线开关后才进行真实训练，不能把NOT_RUN解释成实验通过。
- 所有开发阈值计算先于test读取。训练函数和校准函数在test阶段开始后禁止调用。
- 本文件保留完整指标与不可达阈值状态；不通过挑选某个窗口或种子抹去负结果。
- 本次编写验证不等于全部历史实验再次执行；具体已验证范围见下方交付验证说明。


**交付验证与审核修复（2026-10-20）：** 默认设置已在仅放原始CSV的新内核中执行通过，0次模型拟合、0条test读取。此前的最终数值检查载入了24个既有开发模型，核对代表配置的最终val预测（早期校准基模型只使用val之前训练段的80%前缀），另核对144组前向映射和卡尔曼递推；这不是一次重新训练24个模型的记录。

本次审核修复验证重放79个不平衡配置×5个开发窗的已保存概率，检查1185个本窗召回阈值、948个前窗迁移阈值及相应混淆矩阵；实际运行器确认最终val模型保留D4迁移阈值，单跑val时不会伪造前窗阈值。另载入4个已有模型，验证参照类别/年龄对照、人群切片和Top-k诊断会自动生成，并与原开发记录核对。检查了HGB显式参数、BRF可选包精确版本拒绝逻辑及浮点饱和反例。

**本次新增基模型拟合为0；这些检查不等于五折全量重新训练。全量训练与test重跑均为NOT_RUN。**BRF版本检查使用原隔离依赖，不包含本次BRF重新训练；使用者仍须安装0.1节声明的可选包。